# Notebook for running the winning model
The winning configuration has the following features:
- Sub event types
- Monthly food prices
- Conflict-only text corpus with PCA applied
- K set to 1
- 4 cross validation splits



In [1]:
from model.train_models import train_evaluate_model
from utils.data_prep import get_clean_combined_data

In [2]:
def run_model(config, params):
    data_sources = [
        src
        for src, include in zip(
            ["food", "rain", "text"],
            [
                config["include_food"],
                config["include_rain"],
                config["include_text"],
            ],
        )
        if include
    ]

    model_data, predictor_cols = get_clean_combined_data(
        data_sources=data_sources,
        k=config["k"],
        event_col=config["event_col"],
        conflict_only_embeddings=config["conflict_only"],
    )

    final_params = {
        **params,
        "k": config["k"],
        "event_col": config["event_col"],
        "n_splits": config["n_splits"],
        "use_pca": config["use_pca"],
    }

    results, best_params, shap_importance, onset_predictions = train_evaluate_model(
        model_data,
        predictor_cols,
        final_params,
        best_params=True,  # skip RandomizedSearchCV
        use_pca=config["use_pca"],
        compute_shap=True,
        shap_sample_size=2000,
        return_onset_predictions=True,
    )
    
    return results, best_params, shap_importance, onset_predictions

In [3]:
def summarise(label, subset):
    n_true_pos = subset["y_true"].sum()
    n_caught = subset[(subset["y_true"] == 1) & (subset["y_pred"] == 1)].shape[0]
    recall = n_caught / n_true_pos if n_true_pos else float("nan")
    n_pred_pos = subset["y_pred"].sum()
    precision = n_caught / n_pred_pos if n_pred_pos else float("nan")
    print(
        f"{label}: {len(subset)} rows, {n_true_pos} true escalations, "
        f"{n_caught} caught -> recall={recall:.3f}, precision={precision:.3f}"
    )


## Pipeline A

In [4]:
# --- Model A: numeric-only baseline (ACLED + food + rain, no text) ---
model_a_config = {
    "include_food": True,
    "include_rain": True,
    "include_text": False,
    "conflict_only": None,
    "k": 1.0,
    "event_col": "sub_event_type",
    "n_splits": 4,
    "use_pca": False,
}

# (acled_sub_food_rain_1_4, onset_aupr=0.3213, active_aupr=0.3693)
model_a_xgb_params = {
    "max_depth": 3,
    "min_child_weight": 5,
    "max_delta_step": 0,
    "gamma": 3,
    "learning_rate": 0.1,
    "subsample": 0.6,
    "colsample_bytree": 1.0,
    "reg_alpha": 2.0,
    "reg_lambda": 1,
    "colsample_bylevel": 0.8,
}

In [5]:
results_a, best_params_a, shap_importance_a, onset_predictions_a = run_model(model_a_config,model_a_xgb_params)
print(results_a)
print(best_params_a)
print(shap_importance_a)

INFO:AcledClient:Access token correctly retrieved.
INFO:AcledClient:Reading data from data/acled/acled_sudan.csv.
INFO:Events processing:Data grouped by sub_event_type
INFO:Events processing:Escalation target set at 1.0 standard deviations above the mean.
INFO:Data preparation:ACLED data processed.
INFO:hdx.api.configuration:No HDX base configuration parameter. Using default base configuration file: /Users/evie.jones/Downloads/msc-final-project/.venv/lib/python3.14/site-packages/hdx/api/hdx_base_configuration.yaml.
INFO:hdx.api.configuration:Loading HDX base configuration from: /Users/evie.jones/Downloads/msc-final-project/.venv/lib/python3.14/site-packages/hdx/api/hdx_base_configuration.yaml
INFO:hdx.api.configuration:No HDX configuration parameter and no configuration file at default path: /Users/evie.jones/.hdx_configuration.yaml.
INFO:hdx.api.configuration:Read only access to HDX: True
INFO:HDXClient:Reading local file: data/hdx/hdx_sudan_food_prices.csv
INFO:Data preparation:Food 

--- Fold 1 ---
Train window: 2018-01 to 2018-12 (216 rows)
Test window:  2019-01 to 2019-12 (216 rows)
------------------------------
--- Fold 2 ---
Train window: 2018-01 to 2019-12 (432 rows)
Test window:  2020-01 to 2020-12 (216 rows)
------------------------------
--- Fold 3 ---
Train window: 2018-01 to 2020-12 (648 rows)
Test window:  2021-01 to 2021-12 (216 rows)
------------------------------
--- Fold 4 ---
Train window: 2018-01 to 2021-12 (864 rows)
Test window:  2022-01 to 2022-12 (216 rows)
------------------------------
{'optimal_threshold': '0.3432', 'n_predictors': 31, 'onset_aupr': '0.3213', 'onset_precision_class1': '0.3178', 'onset_recall_class1': '0.5152', 'onset_f1_class1': '0.3931', 'active_aupr': '0.3693', 'active_precision_class1': '0.3246', 'active_recall_class1': '0.6379', 'active_f1_class1': '0.4302'}
{'max_depth': 3, 'min_child_weight': 5, 'max_delta_step': 0, 'gamma': 3, 'learning_rate': 0.1, 'subsample': 0.6, 'colsample_bytree': 1.0, 'reg_alpha': 2.0, 'reg_lam

In [6]:
# Pre/post war-onset split.
# Sudan's civil war broke out 15 April 2023, inside the ONSET test window
# (2023). This checks whether recall differs before vs after that date

onset_predictions_a["year_month"] = onset_predictions_a["year_month"].astype(str)
war_outbreak = "2023-04"

pre_war = onset_predictions_a[onset_predictions_a["year_month"] < war_outbreak]
post_war = onset_predictions_a[onset_predictions_a["year_month"] >= war_outbreak]

summarise("Pre-war  (Jan-Mar 2023)", pre_war)
summarise("Post-war (Apr-Dec 2023)", post_war)

Pre-war  (Jan-Mar 2023): 54 rows, 11 true escalations, 6 caught -> recall=0.545, precision=0.207
Post-war (Apr-Dec 2023): 162 rows, 55 true escalations, 28 caught -> recall=0.509, precision=0.359


## Model B

In [7]:
# --- Model B: numeric + text (winning config) ---
model_b_config = {
    "include_food": True,
    "include_rain": False,
    "include_text": True,
    "conflict_only": True,
    "k": 1.0,
    "event_col": "sub_event_type",
    "n_splits": 4,
    "use_pca": True,
}

# (acled_sub_food_text_conflict_pca_1_4, onset_aupr=0.4214, active_aupr=0.3464)
model_b_xgb_params = {
    "max_depth": 3,
    "min_child_weight": 5,
    "max_delta_step": 0,
    "gamma": 0,
    "learning_rate": 0.01,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 2.0,
    "reg_lambda": 5,
    "colsample_bylevel": 0.6,
}

In [8]:
results, best_params, shap_importance, onset_predictions = run_model(model_b_config,model_b_xgb_params)
print(results)
print(best_params)
print(shap_importance)

INFO:AcledClient:Access token correctly retrieved.
INFO:AcledClient:Reading data from data/acled/acled_sudan.csv.
INFO:Events processing:Data grouped by sub_event_type
INFO:Events processing:Escalation target set at 1.0 standard deviations above the mean.
INFO:Data preparation:ACLED data processed.
INFO:HDXClient:Reading local file: data/hdx/hdx_sudan_food_prices.csv
INFO:Data preparation:Food prices data processed.
INFO:Text processing:Reading local file: data/acled/acled_monthly_regional_embeddings_conflict_only.pkl
INFO:Data preparation:Notes data processed.
INFO:Text processing:PCA fit on train embeddings only: 768 raw dims -> 24 components for 90% variance.
INFO:Cross validation:Cross-validation testing splits:


--- Fold 1 ---
Train window: 2018-01 to 2018-12 (216 rows)
Test window:  2019-01 to 2019-12 (216 rows)
------------------------------
--- Fold 2 ---
Train window: 2018-01 to 2019-12 (432 rows)
Test window:  2020-01 to 2020-12 (216 rows)
------------------------------
--- Fold 3 ---
Train window: 2018-01 to 2020-12 (648 rows)
Test window:  2021-01 to 2021-12 (216 rows)
------------------------------
--- Fold 4 ---
Train window: 2018-01 to 2021-12 (864 rows)
Test window:  2022-01 to 2022-12 (216 rows)
------------------------------
{'optimal_threshold': '0.4407', 'n_predictors': 55, 'onset_aupr': '0.4214', 'onset_precision_class1': '0.3443', 'onset_recall_class1': '0.6364', 'onset_f1_class1': '0.4468', 'active_aupr': '0.3464', 'active_precision_class1': '0.3193', 'active_recall_class1': '0.6552', 'active_f1_class1': '0.4294'}
{'max_depth': 3, 'min_child_weight': 5, 'max_delta_step': 0, 'gamma': 0, 'learning_rate': 0.01, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 2.0, 'reg_la

In [9]:
# Pre/post war-onset split.
# Sudan's civil war broke out 15 April 2023, inside the ONSET test window
# (2023). This checks whether recall differs before vs after that date

onset_predictions["year_month"] = onset_predictions["year_month"].astype(str)
war_outbreak = "2023-04"

pre_war = onset_predictions[onset_predictions["year_month"] < war_outbreak]
post_war = onset_predictions[onset_predictions["year_month"] >= war_outbreak]

summarise("Pre-war  (Jan-Mar 2023)", pre_war)
summarise("Post-war (Apr-Dec 2023)", post_war)

Pre-war  (Jan-Mar 2023): 54 rows, 11 true escalations, 9 caught -> recall=0.818, precision=0.237
Post-war (Apr-Dec 2023): 162 rows, 55 true escalations, 33 caught -> recall=0.600, precision=0.393
